# Stock Analytics Dashboard

This notebook provides comprehensive stock analytics dashboards with interactive visualizations,
statistics, and benchmarking organized by **Phase 9.3 Feature Categories**.

**Version:** 2.1.0 | **Model Version:** v9_10 | **Updated:** 2025-12-26

## Dashboard Structure (21 Feature Categories, 290+ Features)

### 📊 Core Analytics
1. **Region & Sector Overview** - Distribution analysis with feature coverage heatmaps
2. **Exchange Analysis** - Market-level analytics by stock exchange
3. **Industry Deep-Dive** - Granular industry-level benchmarking

### 💹 Financial Analytics (by Feature Category)
4. **Valuation Ratios** - P/E, P/B, EV/EBITDA, PEG analysis
5. **Profitability** - ROE, ROA, ROIC, margin analysis
6. **Growth Metrics** - Revenue, earnings, EBITDA growth
7. **Leverage & Liquidity** - Debt ratios, coverage, liquidity metrics

### 📈 Technical & Market Analytics
8. **Momentum & Technical** - Price momentum, EMA crossovers, breakout signals
9. **Technical Analysis** - RSI, 52-week range, volume momentum (NEW)
10. **Market Sentiment** - Beta stability, systematic risk

### 🎯 Quality & Risk Analytics
11. **Quality & Risk** - Altman Z, accounting quality, distress signals
12. **Composite Scores** - Piotroski F, Beneish M, momentum scores
13. **Earnings Quality** - EPS surprises, GAAP vs Adjusted analytics (NEW)

### 💰 Dividends & Capital Allocation
14. **Dividend Reliability** - Dividend streaks, safety scores, yield analysis (NEW)
15. **Capital Allocation** - CapEx, reinvestment, shareholder returns

### 🔮 Forecasting & Analyst Analytics
16. **Revenue Forecasting** - Estimate spreads, consensus uncertainty (NEW)
17. **Analyst Sentiment** - Price targets, recommendations, conviction

### 👥 Operations & Efficiency
18. **Employee Productivity** - Revenue per employee, hiring intensity
19. **Efficiency Ratios** - Asset turnover, inventory efficiency
20. **Balance Sheet Dynamics** - Asset/debt growth, working capital (NEW)
21. **Employment Dynamics** - FTE growth, workforce volatility (NEW)
22. **Temporal Patterns** - Earnings calendar, fiscal quarters

## Integrated Artifacts
- `outputs/eda/visualizations/` - Sector benchmarking, hypothesis tests
- `outputs/eda/earnings_analytics/` - Earnings surprises, market movers
- `outputs/eda/dividend_visualizations/` - Dividend yield by sector
- `outputs/eda/advanced_analytics/` - VaR, risk attribution


In [1]:
# ============================================================================
# Cell 1: Configuration & Setup
# ============================================================================
import json
import sys
import warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# SQLAlchemy check
try:
    from sqlalchemy import create_engine, text

    HAVE_SQLALCHEMY = True
except ImportError:
    HAVE_SQLALCHEMY = False

# Project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
CACHE_DIR = PROJECT_ROOT / '.cache'

# Create output directories
for subdir in ['eda/dashboards', 'eda/by_region', 'eda/by_sector',
               'eda/by_industry', 'eda/by_exchange']:
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from finance_ml.notebook_config import NotebookConfig
from finance_ml.core.schema import (
    PHASE93_FEATURE_CATEGORIES,
)

# Earnings Dashboard Widgets
from finance_ml.dashboards.widgets import (
    display_earnings_dashboard,
    create_earnings_metrics_chart,
    create_category_comparison_chart,
    create_earnings_surprise_dashboard,
    create_analyst_recommendation_heatmap,
    create_market_movers_dashboard,
    create_price_target_analytics,
    create_earnings_calendar_analytics,
    analyze_earnings_quality,
    create_gaap_adjusted_comparison_chart,
    create_technical_valuation_dashboard,
    create_dividend_sustainability_scorecard,
    create_employee_productivity_dashboard,
    create_category_correlation_network,
    generate_earnings_quality_alerts,
    EarningsAlertConfig,
    resolve_reference_date,
    add_formatted_date_columns,
)
from finance_ml.core.constants import (
    DATE_DISPLAY_FORMAT,
    PLOTLY_TEMPLATE,
    COLOR_PALETTE,
    CATEGORY_COLORS,
)

from finance_ml.features.advanced import engineer_temporal_features
import finance_ml.ml_workflow.eda as eda

CFG = NotebookConfig(
    have_finance_prediction=True,
    have_database_connection=True,
    have_advanced_analytics=True,
    have_dim_reduction=True,
    debug_mode=False,
)

# Configuration Constants
from finance_ml.core.constants import (
    RANDOM_SEED, MODEL_VERSION,
)

np.random.seed(RANDOM_SEED)

# Visualization Configuration
TOP_N_SECTORS = 12
TOP_N_INDUSTRIES = 20
TOP_N_EXCHANGES = 15
TOP_N_FEATURES = 15

# Style Configuration (code_guidelines.md §17)
plt.style.use('dark_background')
sns.set_palette('husl')


def prepare_for_plotly(plotly_df: pd.DataFrame, plotly_columns: list[str] | None = None) -> pd.DataFrame:
    """Convert categorical columns to string for Plotly compatibility."""
    df_copy = plotly_df.copy()
    cols_to_convert = plotly_columns or df_copy.select_dtypes(include=['category']).columns.tolist()
    for col_name in cols_to_convert:
        if col_name in df_copy.columns:
            df_copy[col_name] = df_copy[col_name].astype(str)
    return df_copy


print('=' * 80)
print('STOCK ANALYTICS DASHBOARD - CONFIGURATION')
print('=' * 80)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Python: {sys.version.split()[0]}')
print(f'Model Version: {MODEL_VERSION}')
print(f'Phase 9.3 Categories: {len(PHASE93_FEATURE_CATEGORIES)}')
print(f'Total Phase 9.3 Features: {sum(len(f) for f in PHASE93_FEATURE_CATEGORIES.values())}')
CFG.display_summary()

STOCK ANALYTICS DASHBOARD - CONFIGURATION
PROJECT_ROOT: C:\Users\markm\PycharmProjects\Finance_Analytics_Platform
Python: 3.14.0
Model Version: v9_10
Phase 9.3 Categories: 21
Total Phase 9.3 Features: 303
FEATURE FLAGS CONFIGURATION

Core Features:
  Financial Prediction:        ✓ Enabled
  Database Connection:         ✓ Enabled
  Advanced Analytics:          ✓ Enabled
  Dimensionality Reduction:    ✓ Enabled

Analysis Features:
  Sector Analysis:             ✓ Enabled
  Region Analysis:             ✓ Enabled

Output Features:
  Interactive Plots:           ✓ Enabled
  Excel Export:                ✓ Enabled
  Portfolio Optimization:      ✓ Enabled

Development:
  Debug Mode:                  ✗ Disabled


## Cell 2: ETL Pipeline & Data Loading


In [2]:
# ============================================================================
# Cell 2: ETL Pipeline - Extract, Transform, Load
# ============================================================================

from finance_ml.etl.config import (
    DataExtractionConfig, DataSanitizationConfig, DtypeCastingConfig,
    FeatureEngineeringConfig, FeatureSelectionConfig, FinancialMetricsConfig,
    ImputationConfig, ScalingConfig, SchemaValidationConfig,
    SemanticClassificationConfig, SemanticTransformConfig,
    ETLConfig,
)
from finance_ml.etl.pipeline import run_etl_pipeline
from finance_ml.ml_workflow.eda.phase93_categories import (
    categorize_dataframe_columns, get_phase93_coverage_stats, get_expected_feature_count,
)

etl_config = ETLConfig(
    extraction=DataExtractionConfig(normalize_column_names=True),
    validation=SchemaValidationConfig(
        validate_schema=True, require_target_column=True,
        drop_rows_with_missing_critical_fields=True,
        validate_schema_alignment=True, schema_alignment_threshold=0.80,
    ),
    dtype_casting=DtypeCastingConfig(apply_dtype_casting=True, track_diagnostics=True),
    semantic_classification=SemanticClassificationConfig(enabled=True, preserve_price_columns=True),
    imputation=ImputationConfig(
        apply_imputation=True, strategy="6step", knn_neighbors=5,
        sector_column="sector", reference_price_column="last_price",
        impute_categorical_columns=True, impute_datetime_columns=True,
        # NEW: Business-rule imputation (v1.19)
        apply_dividend_zero_fill=True,
        apply_analyst_rating_zero_fill=True,
        apply_financial_statement_zero_fill=True,
    ),
    semantic_transform=SemanticTransformConfig(
        apply_log_transforms=True, exclude_ratios_from_winsorization=True,
        exclude_percentages_from_winsorization=True,exclude_counts_from_scaling=True
    ),
    sanitization=DataSanitizationConfig(
        sanitize_data=False, apply_winsorization=True,
        # NEW: Business-rule sanitization (v1.19)
        apply_business_rule_zero_fills=True,
    ),
    scaling=ScalingConfig(enabled=False, exclude_price_columns=True),
    feature_engineering=FeatureEngineeringConfig(
        enabled=True, preset="comprehensive", engineer_earnings_analytics=True,
    ),
    feature_selection=FeatureSelectionConfig(enabled=False),
    financial_metrics=FinancialMetricsConfig(
        compute_valuation_metrics=True, compute_profitability_metrics=True,
        compute_growth_metrics=True, compute_leverage_metrics=True,
        compute_target_vs_price_metrics=True,compute_sector_specific_metrics=True,generate_quality_alerts=True,generate_metrics_dashboard=True
    ),
)

print('=' * 80)
print('ETL PIPELINE EXECUTION')
print('=' * 80)

df, metrics = run_etl_pipeline(source='csv', data_dir=DATA_DIR, return_metrics=True, config=etl_config)

print('\n' + metrics.summary())

# Resolve reference date
REFERENCE_DATE = resolve_reference_date(df, None)
print(f"\nReference date: {REFERENCE_DATE.strftime(DATE_DISPLAY_FORMAT)}")

# Add temporal features
if 'next_earnings' in df.columns:
    df = engineer_temporal_features(df, date_col='next_earnings', reference_date=REFERENCE_DATE)

# Add formatted date columns
date_cols = ['_reference_date', 'next_earnings', 'income_statement_report_date', 'dividend_record_ex_date']
date_cols = [c for c in date_cols if c in df.columns]
add_formatted_date_columns(df, date_cols)

# Get Phase 9.3 coverage stats
coverage_stats = get_phase93_coverage_stats(df)
total_phase93 = sum(coverage_stats.values())

print(f'\n✓ ETL Complete: {df.shape[0]:,} stocks × {df.shape[1]} features')
print(f'✓ Phase 9.3 Features: {total_phase93}/{sum(len(f) for f in PHASE93_FEATURE_CATEGORIES.values())}')


ETL PIPELINE EXECUTION


  Affected columns (1): ['price_target_number']...
  Schema status: ['price_target_number (in_schema)']



ETL Pipeline Summary:\n  Source: csv\n  Duration: 13.18s (extract: 0.69s, transform: 12.43s, load: 0.05s)\n  Data: 6914 → 6913 rows, 382 → 623 columns\n  Dtype Casting: Applied (0 coercion warnings, 0 unknown columns) ✓\n  Imputation: 6step (0 → 0 missing) ✓\n  Scaling: None (0 columns) Price Protected: ✓\n  Financial Metrics: 23 added (valuation: 4, profitability: 5, growth: 3, leverage: 2)\n  Semantic Classification: ✓ (Price Columns: 21, Market Value: 21, Ratios: 45, Log-Transformed: 4)\n  Feature Engineering: comprehensive (0 features added)\n  Business Rules: ✓ (0 negative value violations sanitized, 17 log-transforms skipped)\n  Schema Validation: ✓ (alignment: 100.00%, unknown extra: 3, missing required: 0, dtype mismatches: 5, recognized: 477)\n  Quality: 0.997, Validation: 1.000

Reference date: 27 Dec 2025

✓ ETL Complete: 6,913 stocks × 624 features
✓ Phase 9.3 Features: 220/303


In [3]:
df

,ticker,isin,name,description,region,country,trading_country,exchange,unit,sector,...,net_income_adjustment_spread_ltm,net_income_adjustment_pct_ltm,net_income_adjustment_spread_fy,ebitda_adjustment_spread_ltm,ebitda_adjustment_pct_ltm,ebitda_adjustment_spread_fy,ebit_adjustment_spread_ltm,ebit_adjustment_pct_ltm,earnings_quality_warning_flag,days_since_reference
0,NVDA,US67066G1040,NVIDIA Corporation,NVIDIA Corporation a computing infrastructure ...,United States and Canada,US,US,NasdaqGS,USD,Information Technology,...,312.000,0.314522,1385.000,1742.000,1.545751,0.000,6586.000,5.980640,0,60
1,AAPL,US0378331005,Apple Inc.,Apple Inc. designs manufactures and markets sm...,United States and Canada,US,US,NasdaqGS,USD,Information Technology,...,0.000,0.000000,0.000,-253.000,0.174787,-253.000,0.000,0.000000,0,33
2,GOOGL,US02079K3059,Alphabet Inc.,Alphabet Inc. offers various products and plat...,United States and Canada,US,US,NasdaqGS,USD,Communication Services,...,0.000,0.000000,0.000,21896.000,15.082591,20989.000,-1796.000,1.426835,0,39
3,MSFT,US5949181045,Microsoft Corporation,Microsoft Corporation develops and supports so...,United States and Canada,US,US,NasdaqGS,USD,Information Technology,...,3086.000,2.941513,0.000,9331.000,5.606326,6153.000,0.000,0.000000,0,33
4,AMZN,US0231351067,Amazon.com Inc.,Amazon.com Inc. engages in the retail sale of ...,United States and Canada,US,US,NasdaqGS,USD,Consumer Discretionary,...,0.000,0.000000,0.000,22043.000,15.779151,23694.000,-2500.000,3.176580,0,38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6909,LASACO,NGLASACO0002,LASACO Assurance Plc,LASACO Assurance Plc provides various insuranc...,Africa / Middle East,NG,NG,NGSE,NGN,Financials,...,502.680,79790.476190,335.298,532.586,19580.367647,154.358,368.838,11490.280374,0,34
6910,NEIMETH,NGNEIMETH001,Neimeth International Pharmaceuticals Plc,Neimeth International Pharmaceuticals Plc manu...,Africa / Middle East,NG,NG,NGSE,NGN,Health Care,...,69.898,12051.379310,80.124,-33.786,1816.451613,60.862,16.304,926.363636,0,32
6911,TCSA3,BRTCSAACNOR3,Tecnisa S.A.,Tecnisa S.A. develops and constructs residenti...,Latin America and Caribbean,BR,BR,BOVESPA,BRL,Consumer Discretionary,...,-0.020,-0.066445,0.000,22.480,81.804949,19.150,123.618,439.921708,1,83
6912,CILEASING,NGCILEASING2,C & I Leasing Plc,C & I Leasing Plc provides marine services in ...,Africa / Middle East,NG,NG,NGSE,NGN,Industrials,...,240.432,27635.862069,243.020,400.896,2745.863014,388.168,303.076,3297.889010,0,34


## Cell 3: Region & Exchange Distribution Analytics

Interactive dashboards analyzing stock distribution by Region, Sector, Industry, and Exchange.


In [4]:
# ============================================================================
# Cell 3: Region, Sector, Industry & Exchange Distribution
# ============================================================================

print('=' * 80)
print('📊 REGION, SECTOR, INDUSTRY & EXCHANGE ANALYTICS')
print('=' * 80)

dashboard_dir = OUTPUT_DIR / 'eda' / 'dashboards'

# ============================================================================
# 3.1 Region Distribution with Key Metrics
# ============================================================================
if 'region' in df.columns:
    print('\n📍 Region Distribution Analysis...')

    region_summary_stats = df.groupby('region').agg({
        'ticker': 'count',
        'market_cap': ['sum', 'mean', 'median'],
        'last_price': 'mean',
    }).round(2)
    region_summary_stats.columns = ['Count', 'Total_MCap', 'Mean_MCap', 'Median_MCap', 'Avg_Price']
    region_summary_stats = region_summary_stats.sort_values('Count', ascending=False)
    region_summary_stats['Pct'] = (region_summary_stats['Count'] / region_summary_stats['Count'].sum() * 100).round(1)

    # Create region sunburst with sector breakdown
    fig_region_sb = px.sunburst(
        prepare_for_plotly(df, ['region', 'sector']),
        path=['region', 'sector'], values='market_cap',
        title='<b>Market Cap Distribution by Region → Sector</b>',
        template=PLOTLY_TEMPLATE, color='region',
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig_region_sb.update_layout(height=700, font=dict(family='Segoe UI, Roboto, Arial'))
    fig_region_sb.write_html(dashboard_dir / 'region_sector_sunburst.html')
    fig_region_sb.show()

    print(f'  ✓ Saved: region_sector_sunburst.html')
    print(f'\n  Region Summary:')
    print(region_summary_stats.to_string())

# ============================================================================
# 3.2 Exchange Distribution Analysis
# ============================================================================
exchange_col = None
for col in ['exchange', 'primary_exchange', 'stock_exchange']:
    if col in df.columns:
        exchange_col = col
        break

if exchange_col:
    print(f'\n🏛️ Exchange Distribution Analysis ({exchange_col})...')

    exchange_stats = df.groupby(exchange_col).agg({
        'ticker': 'count',
        'market_cap': ['sum', 'mean'],
        'last_price': 'mean',
    }).round(2)
    exchange_stats.columns = ['Count', 'Total_MCap', 'Mean_MCap', 'Avg_Price']
    exchange_stats = exchange_stats.sort_values('Total_MCap', ascending=False).head(TOP_N_EXCHANGES)
    exchange_stats['Pct'] = (exchange_stats['Count'] / df.shape[0] * 100).round(1)

    # Exchange treemap
    top_exchanges = exchange_stats.index.tolist()
    df_exchange = df[df[exchange_col].isin(top_exchanges)]

    fig_exchange = px.treemap(
        prepare_for_plotly(df_exchange, [exchange_col, 'sector']),
        path=[exchange_col, 'sector'],
        values='market_cap', color='market_cap',
        color_continuous_scale='Blues',
        title=f'<b>Market Cap by Exchange → Sector</b><br><sup>Top {TOP_N_EXCHANGES} Exchanges</sup>',
        template=PLOTLY_TEMPLATE,
    )
    fig_exchange.update_layout(height=700)
    fig_exchange.write_html(dashboard_dir / 'exchange_sector_treemap.html')
    fig_exchange.show()

    print(f'  ✓ Saved: exchange_sector_treemap.html')
    print(f'\n  Top Exchanges by Market Cap:')
    print(exchange_stats.head(10).to_string())

# ============================================================================
# 3.3 Industry Distribution Analysis
# ============================================================================
industry_col = None
for col in ['industry', 'industry_group', 'sub_industry']:
    if col in df.columns:
        industry_col = col
        break

if industry_col:
    print(f'\n🏭 Industry Distribution Analysis ({industry_col})...')

    industry_stats = df.groupby(industry_col).agg({
        'ticker': 'count',
        'market_cap': ['sum', 'mean'],
        'roe': 'mean' if 'roe' in df.columns else 'count',
    }).round(2)

    if 'roe' in df.columns:
        industry_stats.columns = ['Count', 'Total_MCap', 'Mean_MCap', 'Avg_ROE']
    else:
        industry_stats.columns = ['Count', 'Total_MCap', 'Mean_MCap', '_count']
        industry_stats = industry_stats.drop('_count', axis=1)

    industry_stats = industry_stats.sort_values('Count', ascending=False).head(TOP_N_INDUSTRIES)

    # Industry bar chart
    fig_industry = px.bar(
        industry_stats.reset_index().head(TOP_N_INDUSTRIES),
        x='Count', y=industry_col, orientation='h',
        title=f'<b>Stock Count by Industry</b><br><sup>Top {TOP_N_INDUSTRIES} Industries</sup>',
        template=PLOTLY_TEMPLATE, color='Total_MCap',
        color_continuous_scale='Viridis',
    )
    fig_industry.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
    fig_industry.write_html(dashboard_dir / 'industry_distribution.html')
    fig_industry.show()

    print(f'  ✓ Saved: industry_distribution.html')

# ============================================================================
# 3.4 Region × Exchange × Sector 3D Analysis
# ============================================================================
if 'region' in df.columns and exchange_col:
    print('\n🌐 Region × Exchange × Sector 3D Analysis...')

    # Create cross-tabulation
    cross_tab = pd.crosstab([df['region'], df['sector']], df[exchange_col])

    # Region-Exchange heatmap
    region_exchange = pd.crosstab(df['region'], df[exchange_col])

    fig_re_heatmap = px.imshow(
        region_exchange, title='<b>Stock Distribution: Region × Exchange</b>',
        template=PLOTLY_TEMPLATE, color_continuous_scale='RdYlGn',
        aspect='auto', text_auto=True,
    )
    fig_re_heatmap.update_layout(height=500)
    fig_re_heatmap.write_html(dashboard_dir / 'region_exchange_heatmap.html')
    fig_re_heatmap.show()

    print(f'  ✓ Saved: region_exchange_heatmap.html')

print('\n✓ Region, Sector, Industry & Exchange Analytics Complete')


📊 REGION, SECTOR, INDUSTRY & EXCHANGE ANALYTICS

📍 Region Distribution Analysis...


  ✓ Saved: region_sector_sunburst.html

  Region Summary:
                             Count   Total_MCap  Mean_MCap  Median_MCap  Avg_Price   Pct
region                                                                                  
Europe                        2058  22205551.94   10789.87      1728.92     203.84  29.8
Asia / Pacific                2053  33759143.70   16443.81      7071.13   13598.16  29.7
United States and Canada      1799  72349783.06   40216.67      6865.94     538.25  26.0
Africa / Middle East           546    923268.94    1690.97      1000.64     185.00   7.9
Latin America and Caribbean    348   1982890.00    5697.96      1865.50    1460.03   5.0
Africa / Middle East           109   3702795.17   33970.60     13102.41     267.43   1.6

🏛️ Exchange Distribution Analysis (exchange)...


  ✓ Saved: exchange_sector_treemap.html

  Top Exchanges by Market Cap:
          Count   Total_MCap  Mean_MCap  Avg_Price   Pct
exchange                                                
NasdaqGS    665  39211626.82   58964.85     119.46   9.6
NYSE       1021  33183436.70   32500.92     862.52  14.8
SEHK        239   6330038.34   26485.52      38.13   3.5
TSE         341   6178813.16   18119.69    5288.54   4.9
SHSE        395   4928217.07   12476.50      60.28   5.7
LSE         317   3696778.97   11661.76      11.47   4.6
ENXTPA      197   3335472.20   16931.33      73.37   2.8
SZSE        312   3230558.29   10354.35      53.58   4.5
TSX         173   3122121.93   18046.95     100.26   2.5
NSEI        191   3100547.62   16233.23    3002.91   2.8

🏭 Industry Distribution Analysis (industry)...


  ✓ Saved: industry_distribution.html

🌐 Region × Exchange × Sector 3D Analysis...


  ✓ Saved: region_exchange_heatmap.html

✓ Region, Sector, Industry & Exchange Analytics Complete


## Cell 4: Phase 9.3 Feature Category Dashboards

Comprehensive dashboards for each of the 21 Phase 9.3 feature categories.


In [5]:
# ============================================================================
# Cell 4: Phase 9.3 Feature Category Dashboards
# ============================================================================

print('=' * 80)
print('📊 PHASE 9.3 FEATURE CATEGORY DASHBOARDS')
print('=' * 80)

category_viz_dir = OUTPUT_DIR / 'eda' / 'dashboards' / 'categories'
category_viz_dir.mkdir(parents=True, exist_ok=True)

# Get categorized features
categorized = categorize_dataframe_columns(df)

# ============================================================================
# 4.1 Category Coverage Overview
# ============================================================================
print('\n📋 Feature Category Coverage Analysis...')

coverage_data = []
for category, features in PHASE93_FEATURE_CATEGORIES.items():
    present = [f for f in features if f in df.columns]
    avg_completeness = 0
    if present:
        avg_completeness = df[present].notna().mean().mean() * 100

    coverage_data.append({
        'Category': category,
        'Expected': len(features),
        'Present': len(present),
        'Coverage_Pct': len(present) / len(features) * 100 if features else 0,
        'Completeness_Pct': avg_completeness,
        'Color': CATEGORY_COLORS.get(category, '#666666'),
    })

coverage_df = pd.DataFrame(coverage_data).sort_values('Expected', ascending=False)

# Coverage bar chart
fig_coverage = go.Figure()
fig_coverage.add_trace(go.Bar(
    y=coverage_df['Category'], x=coverage_df['Expected'],
    name='Expected', orientation='h', marker_color=COLOR_PALETTE['neutral'],
))
fig_coverage.add_trace(go.Bar(
    y=coverage_df['Category'], x=coverage_df['Present'],
    name='Present', orientation='h', marker_color=COLOR_PALETTE['success'],
))
fig_coverage.update_layout(
    title='<b>Phase 9.3 Feature Coverage by Category</b>',
    template=PLOTLY_TEMPLATE, height=700, barmode='overlay',
    xaxis_title='Feature Count', yaxis_title='Category',
)
fig_coverage.write_html(category_viz_dir / 'phase93_coverage_overview.html')
fig_coverage.show()
print(f'  ✓ Saved: phase93_coverage_overview.html')


# ============================================================================
# 4.2 Generate Dashboard for Each Major Category
# ============================================================================

def create_category_dashboard(dashboard_df, category_name, category_features, output_dir, group_cols=None):
    """Create comprehensive dashboard for a feature category."""
    # Filter features to only include numeric columns present in dashboard_df
    available_features = [f for f in category_features
                          if f in dashboard_df.columns and pd.api.types.is_numeric_dtype(dashboard_df[f])]

    if not available_features:
        print(f"  ⚠ No numeric features available for {category_name}")
        return None

    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            f'{category_name}: Distribution by Sector',
            f'{category_name}: Top Features Correlation',
            f'{category_name}: Regional Comparison',
            f'{category_name}: Key Metrics Box Plots',
        ],
        specs=[[{'type': 'bar'}, {'type': 'heatmap'}],
               [{'type': 'scatter'}, {'type': 'box'}]],
        vertical_spacing=0.12, horizontal_spacing=0.1,
    )

    # 1. Mean by Sector (top 10 sectors)
    if 'sector' in dashboard_df.columns and available_features:
        sector_means = dashboard_df.groupby('sector')[available_features[:5]].mean()
        top_sectors = dashboard_df['sector'].value_counts().head(10).index
        sector_means = sector_means.loc[sector_means.index.isin(top_sectors)]

        for i_feat, feat in enumerate(available_features[:3]):
            if feat in sector_means.columns:
                fig.add_trace(
                    go.Bar(x=sector_means.index, y=sector_means[feat], name=feat[:20]),
                    row=1, col=1
                )

    # 2. Correlation heatmap
    if len(available_features) >= 3:
        corr = dashboard_df[available_features[:10]].corr()
        fig.add_trace(
            go.Heatmap(z=corr.values, x=corr.columns, y=corr.index,
                       colorscale='RdBu_r', zmin=-1, zmax=1),
            row=1, col=2
        )

    # 3. Regional scatter (first 2 features)
    if 'region' in dashboard_df.columns and len(available_features) >= 2:
        for region_name in dashboard_df['region'].unique()[:5]:
            region_df_slice = dashboard_df[dashboard_df['region'] == region_name]
            fig.add_trace(
                go.Scatter(x=region_df_slice[available_features[0]], y=region_df_slice[available_features[1]],
                           mode='markers', name=str(region_name)[:15], opacity=0.6),
                row=2, col=1
            )

    # 4. Box plots
    if 'sector' in dashboard_df.columns and available_features:
        for feat in available_features[:2]:
            fig.add_trace(
                go.Box(y=dashboard_df[feat], x=dashboard_df['sector'], name=feat[:15]),
                row=2, col=2
            )

    fig.update_layout(
        title=f'<b>{category_name} Dashboard</b><br><sup>{len(available_features)} features available</sup>',
        template=PLOTLY_TEMPLATE, height=900, showlegend=True,
    )

    output_path_file = output_dir / f'{category_name.lower().replace(" ", "_").replace("&", "and")}_dashboard.html'
    fig.write_html(output_path_file)
    return output_path_file


# Generate dashboards for key categories
key_categories = [
    'Momentum & Technical', 'Valuation Ratios', 'Profitability',
    'Quality & Risk', 'Growth Metrics', 'Leverage & Liquidity',
    'Analyst Sentiment', 'Earnings Quality', 'Dividend Reliability',
    'Revenue Forecasting', 'Balance Sheet Dynamics', 'Technical Analysis',
    'Valuation Timeseries', 'Employment Dynamics',
]

for category in key_categories:
    if category in PHASE93_FEATURE_CATEGORIES:
        features = PHASE93_FEATURE_CATEGORIES[category]
        result = create_category_dashboard(df, category, features, category_viz_dir)
        if result:
            print(f'  ✓ Created: {result.name}')

# ============================================================================
# 4.3 Feature Category Correlation Network
# ============================================================================
print('\n🕸️ Creating Feature Category Correlation Network...')
fig_network = create_category_correlation_network(df, category_mapping=PHASE93_FEATURE_CATEGORIES,
                                                  output_dir=category_viz_dir)
if fig_network is not None:
    fig_network.show()
    print(f'  ✓ Saved: category_correlation_network.html')

print('\n✓ Phase 9.3 Category Dashboards Complete')


📊 PHASE 9.3 FEATURE CATEGORY DASHBOARDS

📋 Feature Category Coverage Analysis...


  ✓ Saved: phase93_coverage_overview.html
  ✓ Created: momentum_and_technical_dashboard.html
  ✓ Created: valuation_ratios_dashboard.html
  ✓ Created: profitability_dashboard.html
  ✓ Created: quality_and_risk_dashboard.html
  ✓ Created: growth_metrics_dashboard.html
  ✓ Created: leverage_and_liquidity_dashboard.html
  ✓ Created: analyst_sentiment_dashboard.html
  ✓ Created: earnings_quality_dashboard.html
  ✓ Created: dividend_reliability_dashboard.html
  ✓ Created: revenue_forecasting_dashboard.html
  ✓ Created: balance_sheet_dynamics_dashboard.html
  ✓ Created: technical_analysis_dashboard.html
  ✓ Created: valuation_timeseries_dashboard.html
  ✓ Created: employment_dynamics_dashboard.html

🕸️ Creating Feature Category Correlation Network...


  ✓ Saved: category_correlation_network.html

✓ Phase 9.3 Category Dashboards Complete


## Cell 5: Regional Benchmarking Dashboard

Comprehensive regional analysis with key financial metrics comparison.


In [6]:
# ============================================================================
# Cell 5: Regional Benchmarking Dashboard
# ============================================================================

print('=' * 80)
print('🌍 REGIONAL BENCHMARKING DASHBOARD')
print('=' * 80)

region_dir = OUTPUT_DIR / 'eda' / 'by_region'

if 'region' in df.columns:
    # Key metrics for regional comparison
    benchmark_metrics = [
        'roe', 'roa', 'p_e_ratio', 'debt_to_equity', 'price_momentum_1m',
        'earnings_quality_score', 'ema_trend_consistency', 'piotroski_f_score',
        'altman_z_score', 'dividend_yield', 'revenue_growth_yoy',
    ]
    available_metrics = [m for m in benchmark_metrics if m in df.columns]

    # ============================================================================
    # 5.1 Regional Statistics Table
    # ============================================================================
    print('\n📊 Computing Regional Statistics...')

    regional_metrics_stats = df.groupby('region')[available_metrics].agg(['mean', 'median', 'std', 'count'])
    regional_metrics_stats.columns = ['_'.join(col) for col in regional_metrics_stats.columns]

    # Save to JSON
    regional_stats_dict = regional_metrics_stats.to_dict()
    with open(region_dir / 'regional_statistics.json', 'w') as f:
        json.dump({k: {str(kk): vv for kk, vv in v.items()}
                   for k, v in regional_stats_dict.items()}, f, indent=2, default=str)
    print(f'  ✓ Saved: regional_statistics.json')

    # ============================================================================
    # 5.2 Regional Radar Charts
    # ============================================================================
    print('\n🎯 Creating Regional Radar Charts...')

    # Normalize metrics for radar chart
    radar_metrics = available_metrics[:8]  # Limit for readability
    regional_means = df.groupby('region')[radar_metrics].mean()

    # Z-score normalize for comparability
    regional_normalized = (regional_means - regional_means.mean()) / regional_means.std()
    regional_normalized = regional_normalized.fillna(0)

    fig_radar = go.Figure()
    for region in regional_normalized.index:
        fig_radar.add_trace(go.Scatterpolar(
            r=regional_normalized.loc[region].values.tolist() + [regional_normalized.loc[region].values[0]],
            theta=radar_metrics + [radar_metrics[0]],
            fill='toself', name=str(region), opacity=0.6,
        ))

    fig_radar.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[-2, 2])),
        title='<b>Regional Financial Profile Comparison</b><br><sup>Z-Score Normalized Metrics</sup>',
        template=PLOTLY_TEMPLATE, height=700,
    )
    fig_radar.write_html(region_dir / 'regional_radar_comparison.html')
    fig_radar.show()
    print(f'  ✓ Saved: regional_radar_comparison.html')

    # ============================================================================
    # 5.3 Regional Box Plots Grid
    # ============================================================================
    print('\n📦 Creating Regional Box Plot Grid...')

    n_metrics = min(6, len(available_metrics))
    rows = 2
    cols = 3

    fig_boxes = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=[m.replace('_', ' ').title() for m in available_metrics[:n_metrics]],
    )

    for idx, metric in enumerate(available_metrics[:n_metrics]):
        row = idx // cols + 1
        col = idx % cols + 1

        for i, region in enumerate(df['region'].unique()):
            region_data = df[df['region'] == region][metric].dropna()
            fig_boxes.add_trace(
                go.Box(y=region_data, name=str(region)[:10],
                       marker_color=px.colors.qualitative.Set2[i % 8],
                       showlegend=(idx == 0)),
                row=row, col=col
            )

    fig_boxes.update_layout(
        title='<b>Key Metrics Distribution by Region</b>',
        template=PLOTLY_TEMPLATE, height=700, showlegend=True,
    )
    fig_boxes.write_html(region_dir / 'regional_boxplots.html')
    fig_boxes.show()
    print(f'  ✓ Saved: regional_boxplots.html')

    # ============================================================================
    # 5.4 Regional Valuation Comparison (Statistical)
    # ============================================================================
    print('\n📊 Performing Statistical Regional Valuation Comparison...')
    valuation_metrics = ['p_e_ratio', 'p_b_ratio', 'ev_ebitda', 'dividend_yield']
    valuation_metrics = [m for m in valuation_metrics if m in df.columns]

    if valuation_metrics:
        regional_val_result = eda.compare_regional_valuations(
            df, metrics=valuation_metrics, region_column='region',
            include_tests=True, test_method='kruskal'
        )

        # The result is a dict with 'distributions' (DataFrame) and 'statistical_tests' (dict)
        distributions_df = regional_val_result['distributions']
        statistical_tests = regional_val_result['statistical_tests']

        # Create an interactive visualization from the distributions data
        fig_regional_val = px.bar(
            distributions_df,
            x='region',
            y='mean',
            color='metric',
            barmode='group',
            error_y='std',
            title='<b>Regional Valuation Comparison</b>',
            template=PLOTLY_TEMPLATE
        )
        fig_regional_val.show()
        fig_regional_val.write_html(region_dir / 'regional_valuation_comparison.html')
        print(f'  ✓ Saved: regional_valuation_comparison.html')

        # Also print statistical significance
        print("\n  📊 Statistical Test Results:")
        for metric, test_result in statistical_tests.items():
            sig = "✓ Significant" if test_result.get('significant') else "✗ Not significant"
            print(f"    {metric}: p-value={test_result.get('p_value', 0):.4f} ({sig})")

print('\n✓ Regional Benchmarking Complete')


🌍 REGIONAL BENCHMARKING DASHBOARD

📊 Computing Regional Statistics...
  ✓ Saved: regional_statistics.json

🎯 Creating Regional Radar Charts...


  ✓ Saved: regional_radar_comparison.html

📦 Creating Regional Box Plot Grid...


  ✓ Saved: regional_boxplots.html

📊 Performing Statistical Regional Valuation Comparison...


  ✓ Saved: regional_valuation_comparison.html

  📊 Statistical Test Results:
    p_e_ratio: p-value=0.0000 (✓ Significant)
    dividend_yield: p-value=0.0000 (✓ Significant)

✓ Regional Benchmarking Complete


## Cell 6: Sector Deep-Dive Analytics


In [7]:
# ============================================================================
# Cell 6: Sector Deep-Dive Analytics
# ============================================================================

print('=' * 80)
print('🏢 SECTOR DEEP-DIVE ANALYTICS')
print('=' * 80)

sector_dir = OUTPUT_DIR / 'eda' / 'by_sector'

if 'sector' in df.columns:
    # ============================================================================
    # 6.1 Sector Performance Heatmap
    # ============================================================================
    print('\n🔥 Creating Sector Performance Heatmap...')

    perf_metrics = [
        'roe', 'roa', 'price_momentum_1m', 'price_momentum_3m',
        'revenue_growth_yoy', 'earnings_quality_score', 'debt_to_equity',
    ]
    perf_metrics = [m for m in perf_metrics if m in df.columns]

    sector_perf = df.groupby('sector')[perf_metrics].mean()

    # Z-score normalize
    sector_perf_z = (sector_perf - sector_perf.mean()) / sector_perf.std()

    fig_heatmap = px.imshow(
        sector_perf_z.T, title='<b>Sector Performance Heatmap</b><br><sup>Z-Score Normalized</sup>',
        template=PLOTLY_TEMPLATE, color_continuous_scale='RdYlGn',
        aspect='auto', text_auto='.2f',
    )
    fig_heatmap.update_layout(height=500, xaxis_title='Sector', yaxis_title='Metric')
    fig_heatmap.write_html(sector_dir / 'sector_performance_heatmap.html')
    fig_heatmap.show()
    print(f'  ✓ Saved: sector_performance_heatmap.html')

    # ============================================================================
    # 6.2 Sector Feature Category Coverage
    # ============================================================================
    print('\n📋 Analyzing Feature Coverage by Sector...')

    sector_coverage = []
    for sector in df['sector'].unique():
        sector_df = df[df['sector'] == sector]
        for category in list(PHASE93_FEATURE_CATEGORIES.keys())[:10]:  # Top 10 categories
            features = PHASE93_FEATURE_CATEGORIES[category]
            available = [f for f in features if f in sector_df.columns]
            if available:
                completeness = sector_df[available].notna().mean().mean() * 100
                sector_coverage.append({
                    'Sector': sector,
                    'Category': category,
                    'Completeness': completeness,
                })

    coverage_matrix = pd.DataFrame(sector_coverage).pivot(
        index='Category', columns='Sector', values='Completeness'
    )

    fig_coverage = px.imshow(
        coverage_matrix,
        title='<b>Phase 9.3 Feature Completeness by Sector × Category</b>',
        template=PLOTLY_TEMPLATE, color_continuous_scale='RdYlGn',
        aspect='auto', text_auto='.0f',
    )
    fig_coverage.update_layout(height=600)
    fig_coverage.write_html(sector_dir / 'sector_category_coverage.html')
    fig_coverage.show()
    print(f'  ✓ Saved: sector_category_coverage.html')

    # ============================================================================
    # 6.3 Sector Valuation Scatter
    # ============================================================================
    print('\n💰 Creating Sector Valuation Scatter...')

    if 'p_e_ratio' in df.columns and 'roe' in df.columns:
        fig_scatter = px.scatter(
            df[df['p_e_ratio'].between(0, 100) & df['roe'].between(-50, 100)],
            x='roe', y='p_e_ratio', color='sector',
            size='market_cap', size_max=30,
            hover_data=['ticker', 'last_price'],
            title='<b>Sector Valuation: ROE vs P/E Ratio</b><br><sup>Size = Market Cap</sup>',
            template=PLOTLY_TEMPLATE,
        )
        fig_scatter.update_layout(height=700)
        fig_scatter.write_html(sector_dir / 'sector_valuation_scatter.html')
        fig_scatter.show()
        print(f'  ✓ Saved: sector_valuation_scatter.html')

    # ============================================================================
    # 6.4 Sector Distribution Comparison
    # ============================================================================
    print('\n📊 Comparing Sector Distributions...')
    dist_metrics = available_metrics.copy()
    dist_metrics = [m for m in dist_metrics if m in df.columns]

    if dist_metrics:
        # Get distribution statistics (returns DataFrame, not Figure)
        sector_dist_df = eda.compare_sector_distributions(df, metrics=dist_metrics, sector_column='sector')

        # Create a visualization from the distribution data
        import plotly.express as px

        fig_sector_dist = px.box(
            df.melt(id_vars=['sector'], value_vars=dist_metrics, var_name='metric', value_name='value'),
            x='sector',
            y='value',
            color='metric',
            facet_col='metric',
            facet_col_wrap=3,
            title='Sector Distribution Comparison'
        )
        fig_sector_dist.update_layout(showlegend=False)
        fig_sector_dist.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

        fig_sector_dist.show()
        fig_sector_dist.write_html(sector_dir / 'sector_distribution_comparison.html')
        print(f'  ✓ Saved: sector_distribution_comparison.html')

print('\n✓ Sector Deep-Dive Analytics Complete')


🏢 SECTOR DEEP-DIVE ANALYTICS

🔥 Creating Sector Performance Heatmap...


  ✓ Saved: sector_performance_heatmap.html

📋 Analyzing Feature Coverage by Sector...


  ✓ Saved: sector_category_coverage.html

💰 Creating Sector Valuation Scatter...


  ✓ Saved: sector_valuation_scatter.html

📊 Comparing Sector Distributions...


  ✓ Saved: sector_distribution_comparison.html

✓ Sector Deep-Dive Analytics Complete


## Cell 7: Earnings & Dividend Analytics (earnings_widgets.py Integration)

Comprehensive earnings and dividend dashboards using Phase 9.3 widgets.


In [8]:
# ============================================================================
# Cell 7: Earnings & Dividend Analytics
# ============================================================================

print('=' * 80)
print('📈 EARNINGS & DIVIDEND ANALYTICS')
print('=' * 80)

earnings_dir = OUTPUT_DIR / 'eda' / 'earnings_analytics'
dividend_dir = OUTPUT_DIR / 'eda' / 'dividend_visualizations'

# ============================================================================
# 7.1 Earnings Surprise Dashboard
# ============================================================================
print('\n📊 Creating Earnings Surprise Dashboard...')
fig_surprise = create_earnings_surprise_dashboard(df, reference_date=REFERENCE_DATE)
if fig_surprise is not None:
    fig_surprise.write_html(earnings_dir / 'earnings_surprise_analysis.html')
    fig_surprise.show()
    print(f'  ✓ Saved: earnings_surprise_analysis.html')

# ============================================================================
# 7.2 Analyst Recommendation Heatmap
# ============================================================================
print('\n🎯 Creating Analyst Recommendation Heatmap...')
fig_analyst = create_analyst_recommendation_heatmap(df)
if fig_analyst is not None:
    fig_analyst.write_html(earnings_dir / 'analyst_recommendations.html')
    fig_analyst.show()
    print(f'  ✓ Saved: analyst_recommendations.html')

# ============================================================================
# 7.3 Market Movers Dashboard
# ============================================================================
print('\n📈 Creating Market Movers Dashboard...')
fig_movers = create_market_movers_dashboard(df, reference_date=REFERENCE_DATE)
if fig_movers is not None:
    fig_movers.write_html(earnings_dir / 'market_movers.html')
    fig_movers.show()
    print(f'  ✓ Saved: market_movers.html')

# ============================================================================
# 7.4 Price Target Analytics
# ============================================================================
print('\n🎯 Creating Price Target Analytics...')
fig_targets = create_price_target_analytics(df)
if fig_targets is not None:
    fig_targets.write_html(earnings_dir / 'price_target_analysis.html')
    fig_targets.show()
    print(f'  ✓ Saved: price_target_analysis.html')

# ============================================================================
# 7.5 Technical & Valuation Dashboard
# ============================================================================
print('\n📊 Creating Technical Valuation Dashboard...')
result_tech = create_technical_valuation_dashboard(df, output_dir=earnings_dir)
if result_tech is not None:
    fig_tech = result_tech['figure']
    fig_tech.show()
    print(f"  ✓ Saved: {result_tech['output_path'].name}")

# ============================================================================
# 7.6 Dividend Sustainability Scorecard
# ============================================================================
print('\n💰 Creating Dividend Sustainability Scorecard...')
output_path_div = dividend_dir / 'dividend_scorecard.html'
fig_div = create_dividend_sustainability_scorecard(df, output_path=output_path_div)
if fig_div is not None and not fig_div.empty:
    print(f"  ✓ Saved: {output_path_div.name}")
    print(f'  ✓ Scorecard contains {len(fig_div)} stocks')

# ============================================================================
# 7.7 Earnings Quality Alerts
# ============================================================================
print('\n⚠️ Generating Earnings Quality Alerts...')
alert_config = EarningsAlertConfig()
alerts_payload = generate_earnings_quality_alerts(df, config=alert_config, reference_date=REFERENCE_DATE)
with open(earnings_dir / 'earnings_alerts.json', 'w') as f:
    json.dump(alerts_payload, f, indent=2, default=str)
print(f"  ✓ Saved: earnings_alerts.json ({len(alerts_payload.get('alerts', []))} alerts)")

# ============================================================================
# 7.8 Earnings Calendar Dashboard (Styled Table)
# ============================================================================
print('\n📅 Creating Styled Earnings Calendar Dashboard...')
styler_dashboard = display_earnings_dashboard(df, mode="all", reference_date=REFERENCE_DATE, top_n=20)
if styler_dashboard is not None:
    # In Jupyter, displaying the styler shows the formatted table
    from IPython.display import display

    display(styler_dashboard)

# ============================================================================
# 7.9 Earnings Metrics Chart
# ============================================================================
print('\n📊 Creating Earnings Metrics Chart...')
fig_metrics = create_earnings_metrics_chart(df, metric_category="Profitability", reference_date=REFERENCE_DATE)
if fig_metrics is not None:
    fig_metrics.write_html(earnings_dir / 'earnings_metrics_profitability.html')
    fig_metrics.show()
    print(f'  ✓ Saved: earnings_metrics_profitability.html')

# ============================================================================
# 7.10 Comprehensive Earnings Calendar Analytics
# ============================================================================
print('\n📅 Generating Comprehensive Earnings Calendar Analytics...')
calendar_results = create_earnings_calendar_analytics(df, output_dir=earnings_dir, reference_date=REFERENCE_DATE)
if calendar_results is not None:
    if 'timeline_fig' in calendar_results:
        calendar_results['timeline_fig'].show()
    if 'heatmap_fig' in calendar_results:
        calendar_results['heatmap_fig'].show()
    print(f"  ✓ Generated earnings calendar analytics")

# ============================================================================
# 7.11 Earnings Quality Analytics
# ============================================================================
print('\n📊 Analyzing Earnings Quality...')
quality_metrics = analyze_earnings_quality(df)
if quality_metrics is not None and not quality_metrics.empty:
    print(f"  ✓ Quality analysis complete: {len(quality_metrics)} indicators")

# ============================================================================
# 7.12 GAAP vs Adjusted Comparison
# ============================================================================
print('\n📊 Creating GAAP vs Adjusted Comparison Chart...')
output_path_gaap = earnings_dir / 'gaap_vs_adjusted_comparison.html'
fig_gaap = create_gaap_adjusted_comparison_chart(df, output_path=output_path_gaap)
if fig_gaap is not None:
    fig_gaap.show()
    print(f'  ✓ Saved: gaap_vs_adjusted_comparison.html')

# ============================================================================
# 7.13 Category Comparison Chart
# ============================================================================
print('\n📊 Creating Category Comparison Chart...')
fig_cat_comp = create_category_comparison_chart(df, reference_date=REFERENCE_DATE)
if fig_cat_comp is not None:
    fig_cat_comp.show()
    fig_cat_comp.write_html(earnings_dir / 'category_comparison_chart.html')
    print(f'  ✓ Saved: category_comparison_chart.html')

print('\n✓ Earnings & Dividend Analytics Complete')


📈 EARNINGS & DIVIDEND ANALYTICS

📊 Creating Earnings Surprise Dashboard...


  ✓ Saved: earnings_surprise_analysis.html

🎯 Creating Analyst Recommendation Heatmap...


  ✓ Saved: analyst_recommendations.html

📈 Creating Market Movers Dashboard...


  ✓ Saved: market_movers.html

🎯 Creating Price Target Analytics...


  ✓ Saved: price_target_analysis.html

📊 Creating Technical Valuation Dashboard...


...


📊 Creating Earnings Metrics Chart...


  ✓ Saved: earnings_metrics_profitability.html

📅 Generating Comprehensive Earnings Calendar Analytics...


  ✓ Generated earnings calendar analytics

📊 Analyzing Earnings Quality...
  ✓ Quality analysis complete: 6913 indicators

📊 Creating GAAP vs Adjusted Comparison Chart...


  ✓ Saved: gaap_vs_adjusted_comparison.html

📊 Creating Category Comparison Chart...


  ✓ Saved: category_comparison_chart.html

✓ Earnings & Dividend Analytics Complete


## Cell 8: Employee Productivity & Efficiency Analytics


In [9]:
# ============================================================================
# Cell 8: Employee Productivity & Efficiency Analytics
# ============================================================================

print('=' * 80)
print('👥 EMPLOYEE PRODUCTIVITY & EFFICIENCY ANALYTICS')
print('=' * 80)

employment_dir = OUTPUT_DIR / 'eda' / 'employment_analytics'

# ============================================================================
# 8.1 Employee Productivity Dashboard
# ============================================================================
print('\n📊 Creating Employee Productivity Dashboard...')
result_emp = create_employee_productivity_dashboard(df, output_dir=employment_dir)
if result_emp is not None:
    fig_emp = result_emp['figure']
    fig_emp.show()
    print(f"  ✓ Saved: {result_emp['output_path'].name}")

# ============================================================================
# 8.2 Efficiency Metrics by Sector
# ============================================================================
efficiency_metrics = [
    'asset_turnover', 'inventory_turnover', 'receivables_turnover',
    'revenue_per_employee', 'profit_per_employee',
]
efficiency_available = [m for m in efficiency_metrics if m in df.columns]

if efficiency_available and 'sector' in df.columns:
    print('\n⚙️ Analyzing Efficiency Metrics by Sector...')

    sector_efficiency = df.groupby('sector')[efficiency_available].mean()

    fig_eff = px.bar(
        sector_efficiency.reset_index().melt(id_vars='sector', var_name='Metric', value_name='Value'),
        x='sector', y='Value', color='Metric', barmode='group',
        title='<b>Efficiency Metrics by Sector</b>',
        template=PLOTLY_TEMPLATE,
    )
    fig_eff.update_layout(height=500, xaxis_tickangle=45)
    fig_eff.write_html(employment_dir / 'efficiency_by_sector.html')
    fig_eff.show()
    print(f'  ✓ Saved: efficiency_by_sector.html')

print('\n✓ Employee Productivity & Efficiency Analytics Complete')


👥 EMPLOYEE PRODUCTIVITY & EFFICIENCY ANALYTICS

📊 Creating Employee Productivity Dashboard...


  ✓ Saved: employee_productivity.html

⚙️ Analyzing Efficiency Metrics by Sector...


  ✓ Saved: efficiency_by_sector.html

✓ Employee Productivity & Efficiency Analytics Complete


## Cell 9: Hypothesis Testing & Statistical Benchmarking


In [10]:
# ============================================================================
# Cell 9: Hypothesis Testing & Statistical Benchmarking
# ============================================================================

print('=' * 80)
print('📊 HYPOTHESIS TESTING & STATISTICAL BENCHMARKING')
print('=' * 80)

import scipy.stats as scipy_stats

stats_dir = OUTPUT_DIR / 'eda' / 'advanced_analytics'

# ============================================================================
# 9.1 ANOVA Tests by Sector
# ============================================================================
test_metrics = ['roe', 'price_momentum_1m', 'debt_to_equity', 'earnings_quality_score']
test_metrics = [m for m in test_metrics if m in df.columns]

if test_metrics and 'sector' in df.columns:
    print('\n📈 Running ANOVA Tests by Sector...')

    anova_results = []
    for metric in test_metrics:
        groups = [group[metric].dropna().values for name, group in df.groupby('sector')]
        groups = [g for g in groups if len(g) >= 5]  # Min samples

        if len(groups) >= 2:
            f_stat, p_value = scipy_stats.f_oneway(*groups)
            anova_results.append({
                'Metric': metric,
                'F_Statistic': round(f_stat, 4),
                'P_Value': round(p_value, 6),
                'Significant': p_value < 0.05,
            })

    anova_df = pd.DataFrame(anova_results)
    print(anova_df.to_string(index=False))

    # Save results
    anova_df.to_json(stats_dir / 'anova_by_sector.json', orient='records', indent=2)
    print(f'  ✓ Saved: anova_by_sector.json')

# ============================================================================
# 9.2 Comprehensive Hypothesis Testing (eda_utils Integration)
# ============================================================================
if test_metrics and 'sector' in df.columns:
    print('\n🧪 Running Comprehensive Hypothesis Tests...')
    hypothesis_results = eda.perform_comprehensive_hypothesis_tests(
        df, metrics=test_metrics, group_column='sector', alpha=0.05
    )

    # Display results
    if 'test_results' in hypothesis_results:
        results_df = pd.DataFrame(hypothesis_results['test_results'])
        print(results_df[['metric', 'test_name', 'p_value', 'is_significant']])

        # Save detailed results
        results_df.to_json(stats_dir / 'comprehensive_hypothesis_tests.json', orient='records', indent=2)
        print(f'  ✓ Saved: comprehensive_hypothesis_tests.json')

    # Hypothesis Test Heatmap (using eda utility results)
    print('\n🔥 Creating Hypothesis Test Heatmap...')
    if 'test_results' in hypothesis_results:
        # Create visualization from results
        viz_metrics = [r['metric'] for r in hypothesis_results['test_results']]
        viz_pvals = [r['p_value'] for r in hypothesis_results['test_results']]

        fig_hyp = go.Figure(data=go.Heatmap(
            z=[viz_pvals],
            x=viz_metrics, y=['Hypothesis Test'],
            colorscale='RdYlGn_r', zmin=0, zmax=0.1,
            text=[[f"p={p:.4f}" for p in viz_pvals]],
            texttemplate='%{text}', textfont={'size': 12},
        ))
        fig_hyp.update_layout(
            title='<b>Hypothesis Test Results: Sector Differences</b><br><sup>Green = Significant Difference</sup>',
            template=PLOTLY_TEMPLATE, height=300,
        )
        fig_hyp.write_html(stats_dir / 'hypothesis_test_heatmap_enhanced.html')
        fig_hyp.show()
        print(f'  ✓ Saved: hypothesis_test_heatmap_enhanced.html')

print('\n✓ Hypothesis Testing Complete')


📊 HYPOTHESIS TESTING & STATISTICAL BENCHMARKING

📈 Running ANOVA Tests by Sector...
                Metric  F_Statistic  P_Value  Significant
                   roe       0.6064 0.889976        False
     price_momentum_1m       9.7795 0.000000         True
        debt_to_equity       0.7238 0.781178        False
earnings_quality_score       4.3947 0.000000         True
  ✓ Saved: anova_by_sector.json

🧪 Running Comprehensive Hypothesis Tests...

🔥 Creating Hypothesis Test Heatmap...

✓ Hypothesis Testing Complete


## Cell 9.5: Peer Analysis & Metric Trends


In [11]:
# ============================================================================
# Cell 9.5: Peer Analysis & Metric Trends
# ============================================================================

print('=' * 80)
print('🔍 PEER ANALYSIS & METRIC TRENDS')
print('=' * 80)

peer_dir = OUTPUT_DIR / 'eda' / 'benchmarking'
peer_dir.mkdir(parents=True, exist_ok=True)

# Select a few representative tickers for demonstration
if not df.empty:
    # Pick top 3 by market cap
    sample_tickers = df.sort_values('market_cap', ascending=False)['ticker'].head(3).tolist()

    for ticker in sample_tickers:
        print(f'\n👥 Analyzing Peer Group for {ticker}...')

        # 1. Find Peer Group
        peers = eda.find_peer_group(df, ticker=ticker, n_peers=5, criteria='market_cap')
        if not peers.empty:
            print(f"  Found {len(peers)} peers in same sector.")
            print(f"  Peers: {', '.join(peers['ticker'].tolist())}")

        # 2. Compare to Peers
        metrics_to_compare = ['p_e_ratio', 'roe', 'revenue_growth_yoy', 'debt_to_equity']
        metrics_to_compare = [m for m in metrics_to_compare if m in df.columns]

        if metrics_to_compare:
            comparison_results = eda.compare_to_peers(df, ticker=ticker, metrics=metrics_to_compare)
            if comparison_results:
                # compare_to_peers returns a dict, not a Figure. Let's print it or summarize it.
                print(f"    Peer comparison for {ticker} complete.")
                for m, stats in comparison_results.items():
                    print(
                        f"      {m}: Target={stats['target']:.2f}, Peer Avg={stats['peers_mean']:.2f}, Dev={stats['deviation_pct']:.1f}%")

        # 3. Analyze Metric Trend (if date-like columns exist)
        # Note: This usually needs time-series data, but we can demonstrate with available fiscal data
        if 'last_price' in df.columns:
            # Demonstration of trend analysis utility
            # (In a real scenario, this would use a historical dataframe)
            pass

# 4. Generate Benchmarking Report
print('\n📋 Generating Global Benchmarking Report...')
bench_metrics = ['roe', 'p_e_ratio', 'dividend_yield', 'market_cap']
bench_metrics = [m for m in bench_metrics if m in df.columns]

if bench_metrics:
    bench_report = eda.generate_benchmarking_report(
        df, metrics=bench_metrics, include_statistical_tests=True
    )
    # Save report summary
    with open(peer_dir / 'benchmarking_report_summary.json', 'w') as f:
        # Convert non-serializable parts if any
        json.dump({k: str(v) for k, v in bench_report.items() if k != 'figures'}, f, indent=2)
    print(f'  ✓ Saved: benchmarking_report_summary.json')

print('\n✓ Peer Analysis Complete')


🔍 PEER ANALYSIS & METRIC TRENDS

👥 Analyzing Peer Group for NVDA...
  Found 5 peers in same sector.
  Peers: AAPL, MSFT, AVGO, 2330, ORCL
    Peer comparison for NVDA complete.
      p_e_ratio: Target=47.04, Peer Avg=180.37, Dev=-73.9%
      roe: Target=83.43, Peer Avg=58.22, Dev=43.3%
      revenue_growth_yoy: Target=207.18, Peer Avg=26.91, Dev=669.9%
      debt_to_equity: Target=0.09, Peer Avg=1.44, Dev=-93.7%

👥 Analyzing Peer Group for AAPL...
  Found 5 peers in same sector.
  Peers: MSFT, NVDA, AVGO, 2330, ORCL
    Peer comparison for AAPL complete.
      p_e_ratio: Target=36.60, Peer Avg=182.46, Dev=-79.9%
      roe: Target=151.91, Peer Avg=44.53, Dev=241.2%
      revenue_growth_yoy: Target=6.43, Peer Avg=67.06, Dev=-90.4%
      debt_to_equity: Target=1.52, Peer Avg=1.15, Dev=32.1%

👥 Analyzing Peer Group for GOOGL...
  Found 5 peers in same sector.
  Peers: META, 700, NFLX, 941, TMUS
    Peer comparison for GOOGL complete.
      p_e_ratio: Target=30.92, Peer Avg=67.77, Dev=-54.4

## Cell 10: Dashboard Summary & Artifact Index


In [12]:
# ============================================================================
# Cell 10: Dashboard Summary & Artifact Index
# ============================================================================

print('=' * 80)
print('📋 DASHBOARD SUMMARY & ARTIFACT INDEX')
print('=' * 80)

# Collect all generated artifacts
artifact_index = {
    'generated_at': datetime.now().isoformat(),
    'reference_date': REFERENCE_DATE.isoformat(),
    'data_shape': {'rows': df.shape[0], 'columns': df.shape[1]},
    'phase93_coverage': coverage_stats,
    'artifacts': {},
}

# Scan output directories
for subdir in ['dashboards', 'by_region', 'by_sector', 'by_industry', 'by_exchange',
               'earnings_analytics', 'dividend_visualizations', 'employment_analytics',
               'advanced_analytics', 'earnings_visualizations', 'visualizations']:
    dir_path = OUTPUT_DIR / 'eda' / subdir
    if dir_path.exists():
        files = list(dir_path.glob('*.html')) + list(dir_path.glob('*.json'))
        artifact_index['artifacts'][subdir] = [f.name for f in files]

# Save artifact index
with open(OUTPUT_DIR / 'eda' / 'artifact_index.json', 'w') as f:
    json.dump(artifact_index, f, indent=2, default=str)

# Print summary
print('\n📁 Generated Artifacts:')
for category, files in artifact_index['artifacts'].items():
    if files:
        print(f'\n  {category}/:')
        for f in files[:5]:
            print(f'    • {f}')
        if len(files) > 5:
            print(f'    ... and {len(files) - 5} more')

total_artifacts = sum(len(f) for f in artifact_index['artifacts'].values())
print(f'\n✓ Total Artifacts Generated: {total_artifacts}')
print(f'✓ Artifact Index: outputs/eda/artifact_index.json')

# Phase 9.3 Coverage Summary
print('\n' + '=' * 80)
print('📊 PHASE 9.3 FEATURE COVERAGE SUMMARY')
print('=' * 80)

print(f'\nCategories: {len(PHASE93_FEATURE_CATEGORIES)}')
print(f'Total Features Registered: {sum(len(f) for f in PHASE93_FEATURE_CATEGORIES.values())}')
print(f'Features Present in Data: {total_phase93}')
print(f'Overall Coverage: {total_phase93 / sum(len(f) for f in PHASE93_FEATURE_CATEGORIES.values()) * 100:.1f}%')

print('\n✓ Stock Analytics Dashboard Complete!')


📋 DASHBOARD SUMMARY & ARTIFACT INDEX

📁 Generated Artifacts:

  dashboards/:
    • exchange_sector_treemap.html
    • industry_distribution.html
    • region_exchange_heatmap.html
    • region_sector_sunburst.html

  by_region/:
    • regional_boxplots.html
    • regional_radar_comparison.html
    • regional_valuation_comparison.html
    • regional_statistics.json

  by_sector/:
    • sector_category_coverage.html
    • sector_distribution_comparison.html
    • sector_performance_heatmap.html
    • sector_valuation_scatter.html

  earnings_analytics/:
    • analyst_recommendations.html
    • analyst_recommendation_heatmap.html
    • category_comparison_chart.html
    • earnings_calendar.html
    • earnings_density_heatmap.html
    ... and 11 more

  dividend_visualizations/:
    • dividend_scorecard.html
    • dividend_yield_by_sector.html
    • dividend_yield_distribution.html
    • forward_dividend_yield_trend.html
    • shareholder_returns_comparison.html

  employment_analytics/:
 